# 02 — Zero-shot baseline

Transcribes the whole test plan (every test utterance × 9 conditions) with both pretrained
models, without fine-tuning, then compares them: WER / CER tables, paired bootstrap tests
and plots of WER and CER against SNR. *Run All* does everything; nothing has to be
switched between the models.

| Step | What | Output in `/kaggle/working` |
|---|---|---|
| 1 | Load PhoWhisper-small and Qwen3-ASR-0.6B, check a few clean utterances | |
| 2 | Transcribe the test plan with each model | `predictions_<model>_zero-shot.csv`, `run_<model>_zero-shot.json` |
| 3 | Score both, tables and paired bootstrap | `wer_table.csv`, `cer_table.csv`, `compare_wer.csv`, `summary_zero-shot.csv` |
| 4 | Plots | `wer_vs_snr.png`, `cer_vs_snr.png`, `error_kinds.png` |

**Kaggle settings**: *GPU T4*, *Internet on*, inputs `asr-noisy-vi` (data) and
`asr-noisy-vi-code` (code). The models are downloaded from Hugging Face. The full run
takes a few hours: use *Save & Run All*, so it does not depend on the browser tab.

**Dry run**: `LIMIT = 20` transcribes 20 utterances per dataset (540 files per model) in
a few minutes. Rows already written are kept when `LIMIT` is set back to `None`.

**Resume**: predictions are appended after every batch. If a run stops, attach that
version's output as an input (*Add Input → Notebook Output*) and run again: the files are
copied back and only the missing rows are transcribed.

**Dataset without `DATASET.json`**: version 1 of `asr-noisy-vi` was uploaded before the
finalize step of notebook 01. The setup cell detects it and rebuilds `DATASET.json` in
`/tmp/asr-data` (audio folders are linked, not copied), so nothing has to be done by hand.

## 0. Setup

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()


def find_code(root=Path("/kaggle/input")):
    """The attached folder holding the packages (the zip may be extracted one level deeper)."""
    for depth in range(1, 6):
        pattern = "/".join(["*"] * depth) + "/dataprep/__init__.py"
        for hit in sorted(root.glob(pattern)):
            if (hit.parents[1] / "pyproject.toml").is_file():
                return hit.parents[1]
    raise FileNotFoundError("Code not found: attach the Kaggle Dataset 'asr-noisy-vi-code' "
                            "(Add Input → Datasets → Your Datasets).")


if Path("dataprep").exists():  # running from inside the repo (local machine)
    code_dir = Path.cwd()
else:
    code_dir = Path("/tmp/asr-noisy-vi")  # /kaggle/input is read-only; pip needs a copy
    shutil.copytree(find_code(), code_dir, dirs_exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{code_dir}[notebooks]"],
               check=True)
sys.path.insert(0, str(code_dir))
version_file = code_dir / "CODE_VERSION.json"
print("code at", code_dir)
print(version_file.read_text() if version_file.exists() else "running from a git checkout")

In [ ]:
import pandas as pd

from dataprep import (
    code_version,
    describe,
    find_data_root,
    iter_eval_batches,
    load_eval_set,
    require_data,
    set_progress_interval,
)
from evaluation import PredictionWriter, load_predictions, missing_predictions, score, summarize


def rebuild_dataset_info(src, root=Path("/tmp/asr-data")):
    """Workaround for a dataset uploaded without DATASET.json (finalize step not run).

    Links the attached audio folders into ``root``, copies the manifests without the
    per-dataset leftovers, and writes LICENSES.md and DATASET.json with the parameters of
    notebook 01. Remove it once the dataset has a version with DATASET.json.
    """
    from dataprep import write_dataset_info, write_licenses

    root.mkdir(exist_ok=True)
    for sub in ["clean", "noisy", "noise"]:
        if not (root / sub).exists():
            (root / sub).symlink_to(src / sub)
    shutil.copytree(src / "manifests", root / "manifests", dirs_exist_ok=True)
    for name in ["fleurs", "vivos", "vietmed"]:
        for prefix in ["clean", "dropped"]:
            (root / "manifests" / f"{prefix}_{name}.csv").unlink(missing_ok=True)
    config = {  # same values as notebook 01
        "seed": 42, "vivos_dev_speakers_per_gender": 2, "vietmed_test_utterances": 1000,
        "musan_subsets": ["librivox"], "musan_test_fraction": 0.3, "test_snrs": [20, 10, 5, 0],
        "dev_clean_prob": 0.25, "train_clean_prob": 0.25, "train_snr_range": [0.0, 20.0],
        "schedule_epochs": 10,
    }
    write_licenses(root)
    write_dataset_info(root, config, "v1 (DATASET.json rebuilt)")
    return root


try:
    DATA_ROOT = find_data_root()
except FileNotFoundError:
    attached = sorted(p.parents[1] for depth in range(1, 6) for p in Path("/kaggle/input").glob(
        "/".join(["*"] * depth) + "/manifests/clean.csv"))
    if not attached:
        raise
    print(f"{attached[0]} has no DATASET.json: rebuilding it in /tmp/asr-data")
    DATA_ROOT = rebuild_dataset_info(attached[0])
info = require_data(DATA_ROOT, steps=("speech", "plans"))
print(DATA_ROOT)
print(describe(info))
MANIFEST_DIR = DATA_ROOT / "manifests"

## Configuration

In [ ]:
MODELS = ["phowhisper-small", "qwen3-asr-0.6b"]  # transcribed in this order
VARIANT = "zero-shot"
LIMIT = None                # dry run: utterances per dataset (e.g. 20); None = the whole test plan
BATCH_SIZE = 16             # lower it if the GPU runs out of memory
PROGRESS_INTERVAL_S = 60

MODEL_IDS = {
    "phowhisper-small": "vinai/PhoWhisper-small",
    "qwen3-asr-0.6b": "Qwen/Qwen3-ASR-0.6B",
}
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS = {m: OUT_DIR / f"predictions_{m}_{VARIANT}.csv" for m in MODELS}
set_progress_interval(PROGRESS_INTERVAL_S)
print(MODELS, VARIANT, "LIMIT =", LIMIT)

Resume from an earlier run: if an attached input holds a model's predictions file and
none exists here yet, copy it so its rows are not transcribed again.

In [ ]:
for model, path in PREDICTIONS.items():
    if not path.exists() and Path("/kaggle/input").exists():
        # Notebook outputs sit one or two levels below /kaggle/input; the dataset folders
        # are not searched recursively, they hold tens of thousands of files.
        earlier = sorted(p for depth in range(1, 4)
                         for p in Path("/kaggle/input").glob("/".join(["*"] * depth) + "/" + path.name))
        if earlier:
            shutil.copy(earlier[0], path)
            print(model, "resumed from", earlier[0])
    print(model, "existing rows:", len(pd.read_csv(path)) if path.exists() else 0)

## 1. Models

The only model-specific part: each loader returns `transcribe(audios)`, which takes a list
of 16 kHz float32 arrays and returns one string per array, **as the model outputs it**
(no normalization: scoring normalizes both sides the same way). Decoding is greedy and
the language is forced to Vietnamese for both models.

`qwen-asr` is installed first, before `torch` and `transformers` are imported, so both
models run with the same library versions.

In [ ]:
if "qwen3-asr-0.6b" in MODELS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qwen-asr"], check=True)

import gc

import torch
import transformers

assert torch.cuda.is_available(), "Turn on a GPU (Settings → Accelerator → GPU T4)"
DEVICE = "cuda:0"
# bfloat16 runs natively from Ampere (compute capability 8) on. A T4 (7.5) reports it as
# supported but only emulates it, which is slower than float16.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print(torch.cuda.get_device_name(0), DTYPE, "transformers", transformers.__version__)
# Generation warnings repeat at every batch and would hide the progress lines.
transformers.logging.set_verbosity_error()


def load_phowhisper():
    from transformers import WhisperForConditionalGeneration, WhisperProcessor

    processor = WhisperProcessor.from_pretrained(MODEL_IDS["phowhisper-small"])
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_IDS["phowhisper-small"])
    model = model.to(device=DEVICE, dtype=DTYPE).eval()
    model.generation_config.max_length = None  # max_new_tokens below sets the limit
    if getattr(model.generation_config, "lang_to_id", None):
        language = {"language": "vi", "task": "transcribe"}
    else:  # older generation config without language tokens: force the prompt instead
        language = {"forced_decoder_ids": processor.get_decoder_prompt_ids(language="vi",
                                                                           task="transcribe")}

    @torch.inference_mode()
    def transcribe(audios):
        # Whisper pads every input to 30 s; test utterances are at most 30 s.
        features = processor.feature_extractor(audios, sampling_rate=16000,
                                               return_tensors="pt").input_features
        ids = model.generate(features.to(DEVICE, DTYPE), **language, num_beams=1,
                             do_sample=False, max_new_tokens=440)
        return processor.batch_decode(ids, skip_special_tokens=True)

    return transcribe


def load_qwen3_asr():
    from qwen_asr import Qwen3ASRModel

    model = Qwen3ASRModel.from_pretrained(
        MODEL_IDS["qwen3-asr-0.6b"],
        dtype=DTYPE,
        device_map=DEVICE,
        max_inference_batch_size=BATCH_SIZE,
        max_new_tokens=512,  # test utterances are at most 30 s
    )

    def transcribe(audios):
        results = model.transcribe(audio=[(a, 16000) for a in audios],
                                   language=["Vietnamese"] * len(audios))
        return [r.text or "" for r in results]

    return transcribe


LOADERS = {"phowhisper-small": load_phowhisper, "qwen3-asr-0.6b": load_qwen3_asr}


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

Sanity check on a few clean test utterances, for both models **before** the long runs:
a model that outputs nothing or another language is caught in minutes, not hours.

In [ ]:
plan = pd.read_csv(MANIFEST_DIR / "test_plan.csv")
clean = pd.read_csv(MANIFEST_DIR / "clean.csv")
sample = load_eval_set(plan, clean, conditions=["clean"]).groupby("dataset").head(2)
sample = sample.sort_values("duration_s", kind="stable")  # the order of iter_eval_batches
_, sample_audios = next(iter_eval_batches(sample, DATA_ROOT, batch_size=len(sample)))

for model in MODELS:
    transcribe = LOADERS[model]()
    hyps = transcribe(sample_audios)
    print(f"=== {model}")
    for (_, row), hyp in zip(sample.iterrows(), hyps):
        print(f"[{row['dataset']}] REF: {row['text']}\n{'':{len(row['dataset']) + 3}}HYP: {hyp}")
    assert sum(bool(h.strip()) for h in hyps) >= len(hyps) // 2, f"{model} returns empty output"
    del transcribe
    free_gpu()

## 2. Transcribe the test plan

In [ ]:
eval_set = load_eval_set(plan, clean, limit=LIMIT)
print(f"{len(eval_set)} files per model:", eval_set.groupby("dataset").size().to_dict())

for model in MODELS:
    transcribe = LOADERS[model]()
    start = time.time()
    with PredictionWriter(PREDICTIONS[model], model, VARIANT) as writer:
        for rows, audios in iter_eval_batches(writer.pending(eval_set), DATA_ROOT,
                                              batch_size=BATCH_SIZE):
            writer.add(rows, transcribe(audios))
    minutes = (time.time() - start) / 60
    record = {
        "model": model,
        "model_id": MODEL_IDS[model],
        "variant": VARIANT,
        "limit": LIMIT,
        "batch_size": BATCH_SIZE,
        "dtype": str(DTYPE),
        "gpu": torch.cuda.get_device_name(0),
        "transformers": transformers.__version__,
        "dataset": describe(info),
        "code": code_version(code_dir),
        "minutes_this_session": round(minutes, 1),
        "finished_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    }
    (OUT_DIR / f"run_{model}_{VARIANT}.json").write_text(json.dumps(record, indent=2))
    print(f"{model}: done in {minutes:.1f} min")
    del transcribe
    free_gpu()

## 3. Scores

WER and CER are corpus-level, in percent, after the same `normalize_text` on the reference
and the hypothesis (see `docs/evaluation.md`).

In [ ]:
from evaluation import bootstrap_ci, compare, wer_table

predictions = load_predictions(PREDICTIONS.values())
predictions = predictions.merge(eval_set[["utt_id", "condition"]])  # the rows of this run
missing = missing_predictions(predictions, eval_set)
assert missing.empty, f"{len(missing)} rows have no prediction:\n{missing.head()}"
scored = score(predictions, plan, clean)

LABELS = {"phowhisper-small": "PhoWhisper-small", "qwen3-asr-0.6b": "Qwen3-ASR-0.6B"}
DATASETS = ["fleurs", "vivos", "vietmed"]
CONDITIONS = ["clean"] + [f"{t} {snr} dB" for t in ("traffic", "babble") for snr in (20, 10, 5, 0)]


def condition_label(noise_type, snr_db):
    return "clean" if noise_type == "clean" else f"{noise_type} {snr_db:g} dB"


def table(summary, metric):
    """Rows: dataset × condition; columns: models."""
    out = summary.assign(condition=[condition_label(t, s) for t, s in
                                    zip(summary["noise_type"], summary["snr_db"])])
    out = out.pivot_table(index=["dataset", "condition"], columns="model", values=metric)
    out = out.reindex(pd.MultiIndex.from_product([DATASETS, CONDITIONS]))
    return out.rename(columns=LABELS).dropna(how="all")


summary = summarize(scored, by=["model", "variant", "dataset", "noise_type", "snr_db"])
summary.to_csv(OUT_DIR / f"summary_{VARIANT}.csv", index=False)
wer = table(summary, "wer")
cer = table(summary, "cer")
wer_table(scored).to_csv(OUT_DIR / "wer_table.csv")
wer_table(scored, metric="cer").to_csv(OUT_DIR / "cer_table.csv")
print("WER (%)")
wer

In [ ]:
print("CER (%)")
cer

All datasets together (corpus-level over the three test sets): how each model degrades
with noise, and how much worse each condition is than clean speech.

In [ ]:
pooled = summarize(scored, by=["model", "noise_type", "snr_db"])
pooled["condition"] = [condition_label(t, s) for t, s in zip(pooled["noise_type"], pooled["snr_db"])]
pooled_wer = pooled.pivot(index="condition", columns="model", values="wer").reindex(CONDITIONS)
pooled_wer = pooled_wer.rename(columns=LABELS)
pd.concat({"WER": pooled_wer, "WER − clean": pooled_wer - pooled_wer.loc["clean"]}, axis=1).round(2)

Paired bootstrap, Qwen3-ASR minus PhoWhisper, per dataset and condition (1000 resamples,
95% interval). A negative `diff` means Qwen3-ASR makes fewer errors; `significant` means
the interval excludes 0, i.e. the gap is not test-set luck.

In [ ]:
comparison = compare(scored, ("phowhisper-small", VARIANT), ("qwen3-asr-0.6b", VARIANT))
comparison = comparison.rename(columns={"wer_a": "wer_phowhisper", "wer_b": "wer_qwen"})
comparison.to_csv(OUT_DIR / "compare_wer.csv", index=False)
comparison

## 4. Plots

WER and CER against SNR, one panel per dataset and noise type, with 95% bootstrap
intervals. The x axis goes from clean to 0 dB, i.e. from left to right the noise gets
louder. Each panel has its own y scale within a row: babble errors are much higher than
traffic errors.

In [ ]:
import matplotlib.pyplot as plt

# Categorical slots 1 and 2 of the reference palette (validated for colour-vision
# deficiencies); markers differ too, so the models are not told apart by colour alone.
COLORS = {"phowhisper-small": "#2a78d6", "qwen3-asr-0.6b": "#eb6834"}
MARKERS = {"phowhisper-small": "o", "qwen3-asr-0.6b": "s"}
X_LABELS = ["clean", "20", "10", "5", "0"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#8a8984", "axes.titlesize": 11,
                     "font.size": 10, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})


def plot_vs_snr(metric, path):
    ci = bootstrap_ci(scored, by=["model", "dataset", "noise_type", "snr_db"], metric=metric)
    fig, axes = plt.subplots(2, len(DATASETS), figsize=(13, 7), sharex=True, sharey="row")
    for r, noise in enumerate(["traffic", "babble"]):
        for c, dataset in enumerate(DATASETS):
            ax = axes[r, c]
            ends = {}
            for model in MODELS:
                part = ci[(ci["model"] == model) & (ci["dataset"] == dataset)]
                rows = pd.concat([part[part["noise_type"] == "clean"],
                                  part[part["noise_type"] == noise].sort_values("snr_db",
                                                                              ascending=False)])
                y = rows[metric].to_numpy()
                err = [y - rows["ci_low"].to_numpy(), rows["ci_high"].to_numpy() - y]
                ax.errorbar(range(len(y)), y, yerr=err, color=COLORS[model],
                            marker=MARKERS[model], markersize=7, linewidth=2, capsize=3,
                            label=LABELS[model])
                ends[model] = y[-1]
            # Value at 0 dB next to each line; spread apart when the two are close.
            close = max(ends.values()) - min(ends.values()) < 0.06 * max(max(ends.values()), 1)
            for model, value in ends.items():
                dy = (7 if value == max(ends.values()) else -7) if close else 0
                ax.annotate(f"{value:.1f}", (len(X_LABELS) - 1, value), xytext=(8, dy),
                            textcoords="offset points", va="center", fontsize=9,
                            color="#52514e")
            ax.set_title(f"{dataset} · {noise}")
            ax.set_xticks(range(len(X_LABELS)), X_LABELS)
            ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
            ax.set_ylim(bottom=0)
            if c == 0:
                ax.set_ylabel(f"{metric.upper()} (%)")
    fig.supxlabel("SNR (dB) — noisier to the right")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.tight_layout()
    fig.suptitle(f"Zero-shot {metric.upper()} against SNR", y=1.07, fontsize=13)
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.03),
               ncol=len(MODELS), frameon=False)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


plot_vs_snr("wer", OUT_DIR / "wer_vs_snr.png")

In [ ]:
plot_vs_snr("cer", OUT_DIR / "cer_vs_snr.png")

Kinds of word errors, all datasets together: substitutions (a wrong word), deletions (a
missed word) and insertions (an extra word). Babble at low SNR typically raises
insertions or substitutions (the model transcribes the background voices or invents
text), traffic raises deletions. The numbers are in the table below the plot.

In [ ]:
KIND_COLORS = {"sub_pct": "#1baf7a", "del_pct": "#eda100", "ins_pct": "#6250d6"}
KIND_LABELS = {"sub_pct": "substitutions", "del_pct": "deletions", "ins_pct": "insertions"}

fig, axes = plt.subplots(1, len(MODELS), figsize=(13, 4.5), sharey=True)
for ax, model in zip(axes, MODELS):
    part = pooled[pooled["model"] == model].set_index("condition").reindex(CONDITIONS)
    bottom = pd.Series(0.0, index=CONDITIONS)
    for kind, color in KIND_COLORS.items():
        ax.bar(CONDITIONS, part[kind], bottom=bottom, color=color, edgecolor="#fcfcfb",
               linewidth=1.5, width=0.7, label=KIND_LABELS[kind])
        bottom += part[kind]
    for i, total in enumerate(bottom):
        ax.annotate(f"{total:.1f}", (i, total), xytext=(0, 3), textcoords="offset points",
                    ha="center", fontsize=8, color="#52514e")
    ax.set_title(LABELS[model])
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)
axes[0].set_ylabel("WER (%)")
handles, labels = axes[0].get_legend_handles_labels()
fig.tight_layout()
fig.suptitle("Word errors by kind, all datasets", y=1.1, fontsize=13)
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.04), ncol=3,
           frameon=False)
fig.savefig(OUT_DIR / "error_kinds.png", dpi=150, bbox_inches="tight")
plt.show()

pooled.assign(model=pooled["model"].map(LABELS)).pivot(
    index="condition", columns="model", values=["sub_pct", "del_pct", "ins_pct"]).reindex(CONDITIONS)

Worst utterances per model, to spot systematic problems (wrong language, hallucinations,
numbers).

In [ ]:
worst = scored.assign(w_err=scored[["w_sub", "w_del", "w_ins"]].sum(axis=1))
worst.sort_values("w_err", ascending=False).groupby("model").head(5)[
    ["model", "utt_id", "condition", "ref", "hyp_norm", "w_err"]]